# Six shared-RGAT experiments: 2,000 entities, 50 epochs, 10 warmup
A: L1 -> fusion -> same L1 -> L2. B: L1 -> fusion -> L2.
C: L1 -> L2 -> fusion -> same L1 -> same L2.
Each runs with a learned refinement gate and without it. Ungated is U + V;
the residual addition and both text gates remain. Seed 0, scratch initialization.
L1 is RGATConv/ReLU/dropout at every use; L2 has no added activation.
Run Sections 1-4 for setup; Section 6 starts the six actual experiments.
No preliminary training runs automatically. GPU execution is not locally verified.


## 1. CUDA environment and local cache settings


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_CACHE'] = '/content/hf_cache/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/hf_cache/hub'
os.environ.pop('TRANSFORMERS_CACHE', None)
import torch
print('Python:', sys.version, '; PyTorch:', torch.__version__, '; CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('Total GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('Free / total bytes:', torch.cuda.mem_get_info())
print('Keep GPU, packages and numerical policy consistent across all six runs.')


## 2. Drive, repository and dependencies


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SHARED_ROOT = Path('/content/drive/MyDrive/Research/scratch_pipeline_v1')
STAGE_ROOT = Path('/content/drive/MyDrive/Research/shared_rgat_2000')
RUN_TAG = 'shared_rgat_2000_50_10_seed0_v1_run1'  # Use the SAME tag to resume; a NEW tag for changed settings.
RUN_ROOT = STAGE_ROOT / RUN_TAG
RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['RESEARCH_MODEL_BACKUP'] = str(SHARED_ROOT / 'verified_roberta_backup')
REPO_URL = 'https://github.com/Rashmika119/Research-project.git'
REPO_REF = 'feature/phase_03'  # Push this stage first. Replace with its exact commit SHA for resume.
REPO_DIR = Path('/content/Research-project')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Repository has local edits. Preserve them before selecting the research commit.'
subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
assert Path('run_shared_rgat_comparison.py').is_file(), 'Push/select the commit containing the six-run shared RGAT study.'
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Exact repository SHA:', COMMIT)
LOCK = RUN_ROOT / 'dependencies.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(LOCK if LOCK.exists() else Path('requirements.txt'))], check=True)
# The lock covers packages used by run identity. CUDA/Python are provided by Colab:
# a changed runtime must be made compatible, never silently accepted for resume.
import importlib.metadata as metadata
packages = ('torch', 'torch-geometric', 'transformers', 'huggingface-hub', 'safetensors',
            'numpy', 'matplotlib', 'pandas', 'pyyaml', 'nbformat', 'ipython')
versions = {name: metadata.version(name) for name in packages}
print('Package versions:', versions)
fresh_torch = subprocess.check_output([sys.executable, '-c', 'import torch; print(torch.__version__)'], text=True).strip()
assert str(torch.__version__) == fresh_torch, 'PyTorch changed during installation; restart runtime and rerun setup.'
if not LOCK.exists():
    LOCK.write_text(''.join(f'{name}=={version}\n' for name, version in versions.items()))
print('Persistent dependency lock:', LOCK)
print('Verified weights load locally from:', os.environ['HF_HUB_CACHE'])


## 3. Approved experiment configuration


In [ ]:
import json
import yaml
import pandas as pd
from dataclasses import asdict
from datetime import datetime
from IPython.display import display, Image, Markdown
from training.shared_study import CONFIGURATIONS, make_config, expected_budget, verify_manifest
from training.experiment_io import write_json, environment
from training.process import run_logged

SEED = 0
MAX_ENTITIES = 2000
TOTAL_EPOCHS = 50
WARMUP_EPOCHS = 10
TEXT_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512
SKIP_COMPLETED = True
RESUME_INTERRUPTED = True
RUN_STATIC_CHECKS = False  # Optional, CPU-only, no optimizer steps.
assert (SEED, MAX_ENTITIES, TOTAL_EPOCHS, WARMUP_EPOCHS) == (0, 2000, 50, 10)
EXPERIMENT_DIR = RUN_ROOT / 'runs'
MANIFEST = EXPERIMENT_DIR / 'dataset_manifest.json'
RAW_DIR = SHARED_ROOT / 'data' / 'fb15k237'
TEXT_DIR = SHARED_ROOT / 'data' / 'text'
POOLED_CACHE = SHARED_ROOT / 'pooled_text_cache'
LOG_DIR = RUN_ROOT / 'logs' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
LOG_DIR.mkdir(parents=True, exist_ok=True)
protocol = yaml.safe_load(Path('experiments/configs/shared_rgat_1000.yaml').read_text())
assert list(CONFIGURATIONS) == list(protocol['configurations'])
assert (protocol['epochs'], protocol['warmup'], protocol['max_entities'], protocol['seeds']) == (50, 10, 2000, [0])
configs = [make_config(c, SEED, raw_dir=RAW_DIR, text_dir=TEXT_DIR,
                      text_batch_size=TEXT_BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE) for c in CONFIGURATIONS]
display(pd.DataFrame([{'configuration': c.variant, 'warmup_epochs': c.warmup,
                      'integrated_epochs': c.epochs-c.warmup, 'train_batch_size': c.train_batch_size,
                      'eval_every': c.eval_every} for c in configs]))
write_json(LOG_DIR / 'environment.json', environment())
with (LOG_DIR / 'pip_freeze.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)


## 4. Non-training subset and model-file verification
Select 2,000 entities using training facts only, then retain all induced facts
from each original split. All six runs use this exact manifest. Candidate set:
2,000 entities, not the full dataset. Cached text is keyed by subset/token identity.
This cell downloads/verifies inputs but performs no forward/backward or training.


In [ ]:
from training.experiment_io import prepare_data
from models.pretrained import prepare_pretrained

data, graph, known_train, manifest = prepare_data(str(RAW_DIR), MAX_ENTITIES, SEED, MANIFEST)
verify_manifest(manifest)
assert len(graph.edge_index) == 2 * len(data.train)
assert set(data.train) == set(known_train)
print('Subset statistics:', manifest['statistics'])
print('Dataset SHA256:', manifest['dataset_sha256'])
print('Manifest SHA256:', manifest['sha256'])
display(pd.DataFrame([{'configuration': c, **expected_budget(c, len(data.train))} for c in CONFIGURATIONS]))
write_json(RUN_ROOT / 'dataset_statistics.json', manifest['statistics'])
prepared = prepare_pretrained()
write_json(LOG_DIR / 'pretrained_files.json', {'revision': prepared.revision, 'files': prepared.files})
print('Verified pretrained files:', prepared.path)
print('Output directory:', EXPERIMENT_DIR)
del data, graph, known_train, manifest


## 5. Optional non-training developer checks


In [ ]:
if RUN_STATIC_CHECKS:
    run_logged([sys.executable, 'run_static_checks.py'], LOG_DIR / 'static_checks.log')
else:
    print('Optional checks disabled. No preliminary training is required.')


## 6. Execute all six experiments
Each child starts from scratch and runs 10 structural + 40 integrated epochs.
Every epoch prints loss, phase, elapsed time and peak GPU memory. Validation
prints every five epochs. Epochs 5 and 10 structural validation and initial integrated
validation are diagnostic only; eligible selection epochs are 15,20,25,30,35,40,45,50.
The selected checkpoint is evaluated on test. No baseline weights are reused.


In [ ]:
command = [sys.executable, '-u', 'run_shared_rgat_comparison.py',
           '--seed', str(SEED), '--raw-dir', str(RAW_DIR), '--text-dir', str(TEXT_DIR),
           '--cache-dir', str(POOLED_CACHE), '--text-batch-size', str(TEXT_BATCH_SIZE),
           '--eval-batch-size', str(EVAL_BATCH_SIZE), '--output-dir', str(EXPERIMENT_DIR)]
if SKIP_COMPLETED:
    command.append('--skip-completed')
if RESUME_INTERRUPTED:
    command.append('--resume')
run_logged(command, LOG_DIR / 'shared_study.log')


## 7. Checkpoints and disconnection recovery
Rerun setup and Section 6 with exactly the same RUN_TAG, source, packages,
settings and GPU. Completed runs are verified/skipped; partial runs restore
their own atomic epoch-end last.pt, including Adam, RNG streams and selected state.
An interrupted, uncommitted epoch replays. Never reuse full-study or another
configuration's checkpoints. Changed identities require a fresh output directory.


In [ ]:
status = []
for c in CONFIGURATIONS:
    folder = EXPERIMENT_DIR / f'{c}_seed{SEED}'
    path = folder / 'results.json'
    r = json.loads(path.read_text()) if path.exists() else None
    status.append({'configuration': c, 'status': 'complete' if r else ('partial' if (folder/'last.pt').exists() else 'not started'),
                   'best_epoch': r['best_epoch'] if r else None, 'checkpoint': str(folder/'last.pt')})
display(pd.DataFrame(status))


## 8. Final comparison, gate effects and graphs
Selection uses validation MRR, never test. Report A/B/C gated-minus-ungated
scores and runtime/memory. A, B and C apply graph layers 3, 2 and 4 times per
integrated forward respectively; C also warms up two layers rather than one.
These are single-seed descriptive comparisons, with no across-seed error bars.
Subset scores are not directly comparable with full-dataset scores.


In [ ]:
from training.shared_reports import shared_report, PLOTS
paths = [EXPERIMENT_DIR / f'{c}_seed{SEED}' / 'results.json' for c in CONFIGURATIONS]
if all(p.is_file() for p in paths):
    summary = shared_report(EXPERIMENT_DIR, [json.loads(p.read_text()) for p in paths], seed=SEED)
    display(pd.read_csv(EXPERIMENT_DIR / 'comparison.csv').sort_values('validation_MRR', ascending=False))
    display(pd.read_csv(EXPERIMENT_DIR / 'gate_effects.csv'))
    display(Markdown((EXPERIMENT_DIR / 'research_analysis.md').read_text()))
    for filename in PLOTS:
        display(Image(filename=str(EXPERIMENT_DIR / filename)))
else:
    print('Complete all six runs before making a comparison.')
print('Persistent results:', EXPERIMENT_DIR)
print('Logs:', LOG_DIR)
